# WiDS 2026 Wildfire Hybrid Stack

In [1]:

import os
import gc
import re
import json
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.optimize import minimize

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.isotonic import IsotonicRegression

warnings.filterwarnings("ignore")

SEED = 2026
N_SPLITS = 5
N_REPEATS = 5
HORIZONS = [12, 24, 48, 72]
PROB_COLS = [f"prob_{h}h" for h in HORIZONS]
USE_EXTERNAL_SUBMISSION_BLEND = True
EXTERNAL_BLEND_MAX_WEIGHT = 0.10

random.seed(SEED)
np.random.seed(SEED)

try:
    from catboost import CatBoostClassifier, CatBoostRegressor
    CATBOOST_AVAILABLE = True
except Exception:
    CATBOOST_AVAILABLE = False

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False


def find_input_file(filename: str) -> str:
    search_roots = [
        Path("/kaggle/input"),
        Path("/kaggle/working"),
        Path("."),
    ]
    matches = []
    for root in search_roots:
        if root.exists():
            matches.extend(root.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"Could not locate {filename}")
    matches = sorted(matches, key=lambda p: ("/kaggle/input" not in str(p), len(str(p))))
    return str(matches[0])


train_path = find_input_file("train.csv")
test_path = find_input_file("test.csv")
sample_sub_path = find_input_file("sample_submission.csv")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_sub = pd.read_csv(sample_sub_path)

assert "event_id" in train.columns and "event_id" in test.columns
assert "time_to_hit_hours" in train.columns and "event" in train.columns

TARGET_TIME = train["time_to_hit_hours"].astype(float).to_numpy()
TARGET_EVENT = train["event"].astype(int).to_numpy()

raw_feature_cols = [c for c in train.columns if c not in ["event_id", "time_to_hit_hours", "event"]]

full = pd.concat(
    [
        train[["event_id"] + raw_feature_cols].copy(),
        test[["event_id"] + raw_feature_cols].copy(),
    ],
    axis=0,
    ignore_index=True,
)


def safe_div(a, b, fill=np.nan):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    out = np.full_like(a, fill, dtype=float)
    np.divide(a, b, out=out, where=np.abs(b) > 1e-12)
    return out


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    for col, period in [
        ("event_start_hour", 24.0),
        ("event_start_dayofweek", 7.0),
        ("event_start_month", 12.0),
    ]:
        if col in df.columns:
            angle = 2.0 * np.pi * df[col].astype(float) / period
            df[f"{col}_sin"] = np.sin(angle)
            df[f"{col}_cos"] = np.cos(angle)

    if "dist_min_ci_0_5h" in df.columns:
        df["dist_km"] = df["dist_min_ci_0_5h"] / 1000.0
        df["dist_to_5km_m"] = df["dist_min_ci_0_5h"] - 5000.0
        df["dist_to_5km_km"] = df["dist_to_5km_m"] / 1000.0
        df["log1p_dist_min_ci"] = np.log1p(np.clip(df["dist_min_ci_0_5h"], a_min=0, a_max=None))
        df["is_already_within_5km"] = (df["dist_min_ci_0_5h"] <= 5000.0).astype(int)

    if "closing_speed_m_per_h" in df.columns:
        df["closing_speed_pos"] = np.clip(df["closing_speed_m_per_h"], a_min=0, a_max=None)
        df["closing_speed_neg"] = np.clip(-df["closing_speed_m_per_h"], a_min=0, a_max=None)
    else:
        df["closing_speed_pos"] = 0.0
        df["closing_speed_neg"] = 0.0

    if "along_track_speed" in df.columns:
        df["along_track_speed_pos"] = np.clip(df["along_track_speed"], a_min=0, a_max=None)
        df["along_track_speed_neg"] = np.clip(-df["along_track_speed"], a_min=0, a_max=None)
    else:
        df["along_track_speed_pos"] = 0.0
        df["along_track_speed_neg"] = 0.0

    if "radial_growth_rate_m_per_h" in df.columns:
        df["radial_growth_rate_pos"] = np.clip(df["radial_growth_rate_m_per_h"], a_min=0, a_max=None)
    else:
        df["radial_growth_rate_pos"] = 0.0

    if "alignment_abs" not in df.columns:
        df["alignment_abs"] = 0.0
    if "alignment_cos" not in df.columns:
        df["alignment_cos"] = 0.0

    df["wavefront_speed_m_per_h"] = (
        df["radial_growth_rate_pos"]
        + df["along_track_speed_pos"] * (0.5 + 0.5 * df["alignment_abs"])
        + 0.50 * df["closing_speed_pos"]
    )

    if "dist_to_5km_m" in df.columns:
        df["eta_closing_h"] = safe_div(df["dist_to_5km_m"], np.clip(df["closing_speed_pos"], 1.0, None), fill=240.0)
        df["eta_along_h"] = safe_div(df["dist_to_5km_m"], np.clip(df["along_track_speed_pos"], 1.0, None), fill=240.0)
        df["eta_wavefront_h"] = safe_div(df["dist_to_5km_m"], np.clip(df["wavefront_speed_m_per_h"], 1.0, None), fill=240.0)
        df["eta_radial_h"] = safe_div(df["dist_to_5km_m"], np.clip(df["radial_growth_rate_pos"], 1.0, None), fill=240.0)
        for c in ["eta_closing_h", "eta_along_h", "eta_wavefront_h", "eta_radial_h"]:
            df[c] = np.clip(df[c], -24.0, 240.0)
            df[f"{c}_inv"] = 1.0 / (1.0 + np.clip(df[c], a_min=0.0, a_max=None))

    if all(c in df.columns for c in ["dist_min_ci_0_5h", "dist_slope_ci_0_5h", "dist_accel_m_per_h2"]):
        for h in HORIZONS:
            proj = (
                df["dist_min_ci_0_5h"]
                + df["dist_slope_ci_0_5h"] * float(h)
                + 0.5 * df["dist_accel_m_per_h2"] * (float(h) ** 2)
            )
            df[f"projected_dist_{h}h_m"] = proj
            df[f"projected_margin_{h}h_km"] = (proj - 5000.0) / 1000.0
            df[f"projected_hit_{h}h"] = (proj <= 5000.0).astype(int)

    if all(c in df.columns for c in ["area_first_ha", "area_growth_abs_0_5h"]):
        df["area_total_ha_5h"] = np.clip(df["area_first_ha"], 0, None) + np.clip(df["area_growth_abs_0_5h"], 0, None)
        df["log1p_area_total_5h"] = np.log1p(df["area_total_ha_5h"])
    if "area_first_ha" in df.columns:
        df["sqrt_area_first"] = np.sqrt(np.clip(df["area_first_ha"], 0, None))
    if "radial_growth_m" in df.columns and "dist_min_ci_0_5h" in df.columns:
        df["near_miss_margin_m"] = df["dist_min_ci_0_5h"] - 5000.0 - df["radial_growth_m"] - np.clip(df.get("projected_advance_m", 0.0), 0, None)

    if all(c in df.columns for c in ["log1p_area_first", "log1p_growth", "dist_km"]):
        df["threat_gravity"] = (
            (1.0 + np.clip(df["log1p_area_first"], 0, None))
            * (1.0 + np.clip(df["log1p_growth"], 0, None))
            * (1.0 + np.clip(df["alignment_abs"], 0, None))
        ) / (1.0 + np.clip(df["dist_km"], 0, None))
    elif "dist_km" in df.columns:
        df["threat_gravity"] = (1.0 + np.clip(df["alignment_abs"], 0, None)) / (1.0 + np.clip(df["dist_km"], 0, None))

    if "dist_km" in df.columns:
        df["momentum_density"] = df["wavefront_speed_m_per_h"] * (1.0 + df["alignment_abs"]) / (1.0 + np.clip(df["dist_km"], 0, None))
        df["closing_pressure"] = df["closing_speed_pos"] / (1.0 + np.clip(df["dist_km"], 0, None))
        df["distance_velocity_ratio"] = safe_div(df["dist_min_ci_0_5h"], 1.0 + df["wavefront_speed_m_per_h"], fill=np.nan)

    df["growth_alignment_interaction"] = df["radial_growth_rate_pos"] * df["alignment_abs"]
    df["closing_alignment_interaction"] = df["closing_speed_pos"] * df["alignment_abs"]
    df["wavefront_alignment_interaction"] = df["wavefront_speed_m_per_h"] * df["alignment_abs"]

    if "low_temporal_resolution_0_5h" in df.columns:
        df["wavefront_if_good_resolution"] = df["wavefront_speed_m_per_h"] * (1.0 - df["low_temporal_resolution_0_5h"])
        df["closing_if_good_resolution"] = df["closing_speed_pos"] * (1.0 - df["low_temporal_resolution_0_5h"])

    if "spread_bearing_sin" in df.columns and "spread_bearing_cos" in df.columns:
        df["bearing_quadrant"] = np.sign(df["spread_bearing_sin"]) * np.sign(df["spread_bearing_cos"])

    df = df.replace([np.inf, -np.inf], np.nan)

    return df


full = engineer_features(full)

feature_cols = [c for c in full.columns if c != "event_id"]
X = full.iloc[: len(train)][feature_cols].copy()
X_test = full.iloc[len(train):][feature_cols].copy()

# Preserve a consistent numeric matrix for all non-CatBoost models.
for df_ in [X, X_test]:
    for col in df_.columns:
        df_[col] = pd.to_numeric(df_[col], errors="coerce")

feature_names = list(X.columns)


def make_strata(time_arr, event_arr):
    labels = []
    for t, e in zip(time_arr, event_arr):
        if e == 1:
            if t <= 12:
                labels.append("e12")
            elif t <= 24:
                labels.append("e24")
            elif t <= 48:
                labels.append("e48")
            else:
                labels.append("e72")
        else:
            if t < 24:
                labels.append("c24")
            elif t < 48:
                labels.append("c48")
            else:
                labels.append("c72")
    labels = np.asarray(labels)
    vc = pd.Series(labels).value_counts()
    if vc.min() < N_SPLITS:
        return event_arr.astype(str)
    return labels


strata = make_strata(TARGET_TIME, TARGET_EVENT)
cv = list(
    RepeatedStratifiedKFold(
        n_splits=N_SPLITS,
        n_repeats=N_REPEATS,
        random_state=SEED,
    ).split(X, strata)
)


def horizon_valid_mask(time_arr, event_arr, horizon):
    return ~((event_arr == 0) & (time_arr < horizon))


def horizon_target(time_arr, event_arr, horizon):
    return ((event_arr == 1) & (time_arr <= horizon)).astype(int)


def balanced_weights(y):
    y = np.asarray(y).astype(int)
    n = len(y)
    pos = int(y.sum())
    neg = int(n - pos)
    if pos == 0 or neg == 0:
        return np.ones(n, dtype=float)
    w = np.ones(n, dtype=float)
    w[y == 1] = n / (2.0 * pos)
    w[y == 0] = n / (2.0 * neg)
    return w


def harrell_c_index(time_arr, event_arr, risk_arr):
    time_arr = np.asarray(time_arr, dtype=float)
    event_arr = np.asarray(event_arr, dtype=int)
    risk_arr = np.asarray(risk_arr, dtype=float)

    concordant = 0.0
    tied = 0.0
    permissible = 0.0

    n = len(time_arr)
    for i in range(n):
        for j in range(i + 1, n):
            ti, tj = time_arr[i], time_arr[j]
            ei, ej = event_arr[i], event_arr[j]
            ri, rj = risk_arr[i], risk_arr[j]

            if ti == tj:
                if ei == 1 and ej == 0:
                    permissible += 1.0
                    if ri > rj:
                        concordant += 1.0
                    elif ri == rj:
                        tied += 1.0
                elif ei == 0 and ej == 1:
                    permissible += 1.0
                    if rj > ri:
                        concordant += 1.0
                    elif ri == rj:
                        tied += 1.0
                continue

            if ti < tj and ei == 1:
                permissible += 1.0
                if ri > rj:
                    concordant += 1.0
                elif ri == rj:
                    tied += 1.0
            elif tj < ti and ej == 1:
                permissible += 1.0
                if rj > ri:
                    concordant += 1.0
                elif ri == rj:
                    tied += 1.0

    if permissible == 0:
        return 0.5
    return (concordant + 0.5 * tied) / permissible


def weighted_brier_score(time_arr, event_arr, pred_matrix):
    pred_matrix = np.asarray(pred_matrix, dtype=float)
    score = 0.0
    for w, h in zip([0.3, 0.4, 0.3], [24, 48, 72]):
        idx = HORIZONS.index(h)
        valid = horizon_valid_mask(time_arr, event_arr, h)
        y = horizon_target(time_arr, event_arr, h)[valid].astype(float)
        p = np.clip(pred_matrix[valid, idx], 1e-6, 1 - 1e-6)
        score += w * np.mean((p - y) ** 2)
    return float(score)


def hybrid_score(time_arr, event_arr, pred_matrix):
    pred_matrix = np.asarray(pred_matrix, dtype=float)
    pred_matrix = np.maximum.accumulate(np.clip(pred_matrix, 0.0, 1.0), axis=1)
    cidx = harrell_c_index(time_arr, event_arr, pred_matrix[:, 0])
    wbs = weighted_brier_score(time_arr, event_arr, pred_matrix)
    return 0.3 * cidx + 0.7 * (1.0 - wbs)


def enforce_monotonic(pred_matrix):
    pred_matrix = np.asarray(pred_matrix, dtype=float)
    pred_matrix = np.clip(pred_matrix, 0.0, 1.0)
    pred_matrix = np.maximum.accumulate(pred_matrix, axis=1)
    pred_matrix = np.clip(pred_matrix, 0.0, 1.0)
    return pred_matrix


class ConstantProbModel:
    def __init__(self, p):
        self.p = float(np.clip(p, 1e-6, 1 - 1e-6))

    def fit(self, X, y, sample_weight=None):
        return self

    def predict_proba(self, X):
        p = np.full(len(X), self.p, dtype=float)
        return np.column_stack([1.0 - p, p])


class ConstantRegModel:
    def __init__(self, value):
        self.value = float(value)

    def fit(self, X, y, sample_weight=None):
        return self

    def predict(self, X):
        return np.full(len(X), self.value, dtype=float)


def build_cb_classifier(horizon, seed):
    depth = 5 if horizon <= 24 else 6
    iters = 450 if horizon <= 24 else 550
    return CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="Logloss",
        iterations=iters,
        depth=depth,
        learning_rate=0.035,
        l2_leaf_reg=8.0,
        random_strength=0.8,
        bagging_temperature=0.8,
        border_count=128,
        bootstrap_type="Bayesian",
        random_seed=seed,
        verbose=False,
        allow_writing_files=False,
    )


def monotone_constraints_for_features(features):
    mono = {f: 0 for f in features}
    for f in [
        "dist_min_ci_0_5h",
        "dist_to_5km_m",
        "dist_to_5km_km",
        "eta_closing_h",
        "eta_along_h",
        "eta_wavefront_h",
        "eta_radial_h",
        "projected_dist_12h_m",
        "projected_dist_24h_m",
        "projected_dist_48h_m",
        "projected_dist_72h_m",
        "projected_margin_12h_km",
        "projected_margin_24h_km",
        "projected_margin_48h_km",
        "projected_margin_72h_km",
        "distance_velocity_ratio",
    ]:
        if f in mono:
            mono[f] = -1
    for f in [
        "closing_speed_pos",
        "along_track_speed_pos",
        "radial_growth_rate_pos",
        "wavefront_speed_m_per_h",
        "closing_pressure",
        "momentum_density",
    ]:
        if f in mono:
            mono[f] = 1
    return tuple(mono[f] for f in features)


def build_xgb_classifier(horizon, seed, features):
    depth = 3 if horizon <= 24 else 4
    n_estimators = 420 if horizon <= 24 else 520
    return XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_estimators=n_estimators,
        learning_rate=0.03,
        max_depth=depth,
        min_child_weight=2.0,
        subsample=0.85,
        colsample_bytree=0.80,
        reg_alpha=0.25,
        reg_lambda=2.5,
        gamma=0.0,
        max_bin=256,
        tree_method="hist",
        monotone_constraints=monotone_constraints_for_features(features),
        random_state=seed,
        n_jobs=4,
    )


def build_lr_classifier():
    return Pipeline(
        steps=[
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler()),
            ("lr", LogisticRegression(C=0.6, max_iter=4000, solver="lbfgs")),
        ]
    )


def build_hgb_classifier(horizon, seed):
    return HistGradientBoostingClassifier(
        learning_rate=0.045,
        max_depth=3 if horizon <= 24 else 4,
        max_iter=450 if horizon <= 24 else 550,
        min_samples_leaf=8,
        l2_regularization=0.20,
        random_state=seed,
    )


def build_cb_regressor(seed):
    return CatBoostRegressor(
        loss_function="RMSE",
        iterations=550,
        depth=5,
        learning_rate=0.035,
        l2_leaf_reg=8.0,
        random_strength=0.8,
        bagging_temperature=0.8,
        border_count=128,
        bootstrap_type="Bayesian",
        random_seed=seed,
        verbose=False,
        allow_writing_files=False,
    )


def build_hgb_regressor(seed):
    return HistGradientBoostingRegressor(
        learning_rate=0.04,
        max_depth=3,
        max_iter=500,
        min_samples_leaf=8,
        l2_regularization=0.25,
        random_state=seed,
    )



def train_direct_family(name, X_df, X_test_df, time_arr, event_arr, splits, builder):
    n_train = len(X_df)
    n_test = len(X_test_df)
    oof_sum = np.zeros((n_train, len(HORIZONS)), dtype=float)
    oof_count = np.zeros((n_train, len(HORIZONS)), dtype=float)
    test_pred = np.zeros((n_test, len(HORIZONS)), dtype=float)

    for h_idx, horizon in enumerate(HORIZONS):
        fold_test_acc = np.zeros(n_test, dtype=float)
        for fold, (trn_idx, val_idx) in enumerate(splits):
            trn_idx = np.asarray(trn_idx)
            val_idx = np.asarray(val_idx)

            valid_trn = horizon_valid_mask(time_arr[trn_idx], event_arr[trn_idx], horizon)
            fit_idx = trn_idx[valid_trn]

            y_fit = horizon_target(time_arr[fit_idx], event_arr[fit_idx], horizon)
            if len(fit_idx) == 0:
                model = ConstantProbModel(float(horizon_target(time_arr, event_arr, horizon).mean()))
            elif np.unique(y_fit).size < 2:
                model = ConstantProbModel(float(y_fit.mean()))
            else:
                sw = balanced_weights(y_fit)
                if name == "direct_cb":
                    model = builder(horizon, SEED + fold)
                elif name == "direct_xgb":
                    model = builder(horizon, SEED + fold, feature_names)
                elif name == "direct_hgb":
                    model = builder(horizon, SEED + fold)
                else:
                    model = builder()

                try:
                    if name in {"direct_cb", "direct_xgb", "direct_hgb"}:
                        model.fit(X_df.iloc[fit_idx], y_fit, sample_weight=sw)
                    else:
                        model.fit(X_df.iloc[fit_idx], y_fit, lr__sample_weight=sw)
                except TypeError:
                    model.fit(X_df.iloc[fit_idx], y_fit)

            fold_val_pred = model.predict_proba(X_df.iloc[val_idx])[:, 1]
            oof_sum[val_idx, h_idx] += fold_val_pred
            oof_count[val_idx, h_idx] += 1.0
            fold_test_acc += model.predict_proba(X_test_df)[:, 1] / len(splits)

        test_pred[:, h_idx] = fold_test_acc

    oof = oof_sum / np.maximum(oof_count, 1.0)
    return enforce_monotonic(oof), enforce_monotonic(test_pred)


BIN_EDGES = [0, 12, 24, 48, 72]


def expand_discrete_hazard(X_df, time_arr, event_arr):
    rows = []
    periods = []
    labels = []
    weights = []

    for i in range(len(X_df)):
        t = float(time_arr[i])
        e = int(event_arr[i])

        sample_periods = []
        sample_labels = []
        for b in range(4):
            lo = BIN_EDGES[b]
            hi = BIN_EDGES[b + 1]

            if e == 0 and t < hi:
                break

            is_event_bin = int(e == 1 and (t > lo) and (t <= hi))
            sample_periods.append(b)
            sample_labels.append(is_event_bin)

            if is_event_bin == 1:
                break

        k = len(sample_periods)
        if k == 0:
            continue

        for b, lab in zip(sample_periods, sample_labels):
            rows.append(i)
            periods.append(b)
            labels.append(lab)
            weights.append(1.0 / k)

    exp_X = X_df.iloc[rows].copy().reset_index(drop=True)
    exp_X["period"] = periods
    return exp_X, np.asarray(labels, dtype=int), np.asarray(weights, dtype=float)


def hazard_to_cumulative(hazards):
    hazards = np.clip(np.asarray(hazards, dtype=float), 1e-6, 1 - 1e-6)
    return 1.0 - np.cumprod(1.0 - hazards, axis=1)


def predict_hazard_cb(model, X_eval):
    hazards = []
    for b in range(4):
        tmp = X_eval.copy()
        tmp["period"] = str(b)
        hazards.append(model.predict_proba(tmp)[:, 1])
    hazards = np.column_stack(hazards)
    return hazard_to_cumulative(hazards)



def train_hazard_cb_family(X_df, X_test_df, time_arr, event_arr, splits):
    n_train = len(X_df)
    n_test = len(X_test_df)
    oof_sum = np.zeros((n_train, 4), dtype=float)
    oof_count = np.zeros(n_train, dtype=float)
    test_pred = np.zeros((n_test, 4), dtype=float)

    for fold, (trn_idx, val_idx) in enumerate(splits):
        trn_idx = np.asarray(trn_idx)
        val_idx = np.asarray(val_idx)

        X_exp, y_exp, w_exp = expand_discrete_hazard(X_df.iloc[trn_idx].reset_index(drop=True), time_arr[trn_idx], event_arr[trn_idx])
        X_exp["period"] = X_exp["period"].astype(str)

        if len(y_exp) == 0 or np.unique(y_exp).size < 2:
            base_rate = float(y_exp.mean()) if len(y_exp) else 0.05
            fold_oof = np.tile(np.asarray([base_rate, base_rate, base_rate, base_rate]), (len(val_idx), 1))
            fold_test = np.tile(np.asarray([base_rate, base_rate, base_rate, base_rate]), (n_test, 1))
        else:
            sw = w_exp * balanced_weights(y_exp)
            model = CatBoostClassifier(
                loss_function="Logloss",
                eval_metric="Logloss",
                iterations=550,
                depth=5,
                learning_rate=0.035,
                l2_leaf_reg=8.0,
                random_strength=0.8,
                bagging_temperature=0.8,
                border_count=128,
                bootstrap_type="Bayesian",
                random_seed=SEED + fold,
                verbose=False,
                allow_writing_files=False,
            )
            model.fit(X_exp, y_exp, sample_weight=sw, cat_features=["period"])
            fold_oof = predict_hazard_cb(model, X_df.iloc[val_idx])
            fold_test = predict_hazard_cb(model, X_test_df)

        oof_sum[val_idx] += fold_oof
        oof_count[val_idx] += 1.0
        test_pred += fold_test / len(splits)

    oof = oof_sum / np.maximum(oof_count[:, None], 1.0)
    return enforce_monotonic(oof), enforce_monotonic(test_pred)


def hazard_lr_design(df):
    out = df.copy()
    out["period"] = out["period"].astype(int)
    out = pd.get_dummies(out, columns=["period"], prefix="period", drop_first=False)
    for p in range(4):
        col = f"period_{p}"
        if col not in out.columns:
            out[col] = 0
    return out


def predict_hazard_lr(model, cols, X_eval):
    hazards = []
    for b in range(4):
        tmp = X_eval.copy()
        tmp["period"] = b
        tmp = hazard_lr_design(tmp)
        missing = [c for c in cols if c not in tmp.columns]
        for c in missing:
            tmp[c] = 0
        tmp = tmp[cols]
        hazards.append(model.predict_proba(tmp)[:, 1])
    hazards = np.column_stack(hazards)
    return hazard_to_cumulative(hazards)



def train_hazard_lr_family(X_df, X_test_df, time_arr, event_arr, splits):
    n_train = len(X_df)
    n_test = len(X_test_df)
    oof_sum = np.zeros((n_train, 4), dtype=float)
    oof_count = np.zeros(n_train, dtype=float)
    test_pred = np.zeros((n_test, 4), dtype=float)

    for fold, (trn_idx, val_idx) in enumerate(splits):
        trn_idx = np.asarray(trn_idx)
        val_idx = np.asarray(val_idx)

        X_exp, y_exp, w_exp = expand_discrete_hazard(X_df.iloc[trn_idx].reset_index(drop=True), time_arr[trn_idx], event_arr[trn_idx])
        X_exp = hazard_lr_design(X_exp)
        cols = list(X_exp.columns)

        if len(y_exp) == 0 or np.unique(y_exp).size < 2:
            base_rate = float(y_exp.mean()) if len(y_exp) else 0.05
            fold_oof = np.tile(np.asarray([base_rate, base_rate, base_rate, base_rate]), (len(val_idx), 1))
            fold_test = np.tile(np.asarray([base_rate, base_rate, base_rate, base_rate]), (n_test, 1))
        else:
            sw = w_exp * balanced_weights(y_exp)
            model = Pipeline(
                steps=[
                    ("imp", SimpleImputer(strategy="median")),
                    ("sc", StandardScaler()),
                    ("lr", LogisticRegression(C=0.7, max_iter=4000, solver="lbfgs")),
                ]
            )
            model.fit(X_exp, y_exp, lr__sample_weight=sw)
            fold_oof = predict_hazard_lr(model, cols, X_df.iloc[val_idx])
            fold_test = predict_hazard_lr(model, cols, X_test_df)

        oof_sum[val_idx] += fold_oof
        oof_count[val_idx] += 1.0
        test_pred += fold_test / len(splits)

    oof = oof_sum / np.maximum(oof_count[:, None], 1.0)
    return enforce_monotonic(oof), enforce_monotonic(test_pred)



def fit_rank_regression_family(name, X_df, X_test_df, time_arr, event_arr, splits, builder):
    n_train = len(X_df)
    n_test = len(X_test_df)
    oof_sum = np.zeros(n_train, dtype=float)
    oof_count = np.zeros(n_train, dtype=float)
    test_risk = np.zeros(n_test, dtype=float)

    reg_target = np.where(event_arr == 1, time_arr, np.minimum(time_arr + 18.0, 90.0))
    reg_target = np.exp(-reg_target / 24.0)
    reg_weight = np.where(event_arr == 1, 1.0, 0.60)

    for fold, (trn_idx, val_idx) in enumerate(splits):
        trn_idx = np.asarray(trn_idx)
        val_idx = np.asarray(val_idx)

        y_fit = reg_target[trn_idx]
        sw = reg_weight[trn_idx]

        model = builder(SEED + fold)
        try:
            model.fit(X_df.iloc[trn_idx], y_fit, sample_weight=sw)
        except TypeError:
            model.fit(X_df.iloc[trn_idx], y_fit)

        fold_val_pred = model.predict(X_df.iloc[val_idx])
        oof_sum[val_idx] += fold_val_pred
        oof_count[val_idx] += 1.0
        test_risk += model.predict(X_test_df) / len(splits)

    oof_risk = oof_sum / np.maximum(oof_count, 1.0)
    return oof_risk, test_risk


def calibrate_risk_with_isotonic(oof_risk, test_risk, time_arr, event_arr):
    oof = np.zeros((len(oof_risk), 4), dtype=float)
    test_pred = np.zeros((len(test_risk), 4), dtype=float)

    for h_idx, horizon in enumerate(HORIZONS):
        valid = horizon_valid_mask(time_arr, event_arr, horizon)
        y = horizon_target(time_arr, event_arr, horizon)[valid]
        if np.unique(y).size < 2 or np.unique(oof_risk[valid]).size < 3:
            p = float(y.mean()) if len(y) else 0.05
            oof[:, h_idx] = p
            test_pred[:, h_idx] = p
        else:
            iso = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
            iso.fit(oof_risk[valid], y)
            oof[:, h_idx] = iso.predict(oof_risk)
            test_pred[:, h_idx] = iso.predict(test_risk)

    return enforce_monotonic(oof), enforce_monotonic(test_pred)


def beta_calibrate_probs(oof_prob, test_prob, valid_mask, y_valid):
    oof_prob = np.asarray(oof_prob, dtype=float)
    test_prob = np.asarray(test_prob, dtype=float)
    valid_mask = np.asarray(valid_mask, dtype=bool)
    y_valid = np.asarray(y_valid, dtype=int)

    p = np.clip(oof_prob[valid_mask], 1e-6, 1 - 1e-6)
    if np.unique(y_valid).size < 2 or np.unique(p).size < 3:
        return oof_prob, test_prob

    X_cal = np.column_stack([np.log(p), np.log1p(-p)])
    lr = LogisticRegression(C=0.5, max_iter=4000, solver="lbfgs")
    lr.fit(X_cal, y_valid)

    def _pred(arr):
        arr = np.clip(arr, 1e-6, 1 - 1e-6)
        feats = np.column_stack([np.log(arr), np.log1p(-arr)])
        return lr.predict_proba(feats)[:, 1]

    oof_cal = _pred(oof_prob)
    test_cal = _pred(test_prob)

    oof_blend = 0.85 * oof_cal + 0.15 * oof_prob
    test_blend = 0.85 * test_cal + 0.15 * test_prob
    return oof_blend, test_blend


family_oof = {}
family_test = {}

if CATBOOST_AVAILABLE:
    print("Training: direct_cb")
    family_oof["direct_cb"], family_test["direct_cb"] = train_direct_family(
        "direct_cb", X, X_test, TARGET_TIME, TARGET_EVENT, cv, build_cb_classifier
    )

if XGBOOST_AVAILABLE:
    print("Training: direct_xgb")
    family_oof["direct_xgb"], family_test["direct_xgb"] = train_direct_family(
        "direct_xgb", X, X_test, TARGET_TIME, TARGET_EVENT, cv, build_xgb_classifier
    )

print("Training: direct_lr")
family_oof["direct_lr"], family_test["direct_lr"] = train_direct_family(
    "direct_lr", X, X_test, TARGET_TIME, TARGET_EVENT, cv, build_lr_classifier
)

print("Training: direct_hgb")
family_oof["direct_hgb"], family_test["direct_hgb"] = train_direct_family(
    "direct_hgb", X, X_test, TARGET_TIME, TARGET_EVENT, cv, build_hgb_classifier
)

if CATBOOST_AVAILABLE:
    print("Training: hazard_cb")
    family_oof["hazard_cb"], family_test["hazard_cb"] = train_hazard_cb_family(
        X, X_test, TARGET_TIME, TARGET_EVENT, cv
    )

print("Training: hazard_lr")
family_oof["hazard_lr"], family_test["hazard_lr"] = train_hazard_lr_family(
    X, X_test, TARGET_TIME, TARGET_EVENT, cv
)

if CATBOOST_AVAILABLE:
    print("Training: urgency_cb")
    urg_oof_risk, urg_test_risk = fit_rank_regression_family(
        "urgency_cb", X, X_test, TARGET_TIME, TARGET_EVENT, cv, build_cb_regressor
    )
    family_oof["urgency_cb"], family_test["urgency_cb"] = calibrate_risk_with_isotonic(
        urg_oof_risk, urg_test_risk, TARGET_TIME, TARGET_EVENT
    )

print("Training: urgency_hgb")
urg_oof_risk, urg_test_risk = fit_rank_regression_family(
    "urgency_hgb", X, X_test, TARGET_TIME, TARGET_EVENT, cv, build_hgb_regressor
)
family_oof["urgency_hgb"], family_test["urgency_hgb"] = calibrate_risk_with_isotonic(
    urg_oof_risk, urg_test_risk, TARGET_TIME, TARGET_EVENT
)

family_names = list(family_oof.keys())
stack_oof = np.stack([family_oof[name] for name in family_names], axis=0)   # [M, N, 4]
stack_test = np.stack([family_test[name] for name in family_names], axis=0) # [M, T, 4]

print("\nOOF family scores:")
for name in family_names:
    print(f"{name:>12s} : {hybrid_score(TARGET_TIME, TARGET_EVENT, family_oof[name]):.6f}")


def optimize_simplex(objective, m, shrink=0.20):
    if m == 1:
        return np.ones(1, dtype=float)

    starts = [np.ones(m) / m]
    for i in range(m):
        w = np.zeros(m)
        w[i] = 1.0
        starts.append(w)
    best_val = None
    best_w = starts[0]

    bounds = [(0.0, 1.0)] * m
    cons = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]

    for x0 in starts:
        res = minimize(
            objective,
            x0=x0,
            method="SLSQP",
            bounds=bounds,
            constraints=cons,
            options={"maxiter": 500, "ftol": 1e-9},
        )
        if res.success:
            w = np.clip(res.x, 0.0, 1.0)
            if w.sum() <= 0:
                continue
            w = w / w.sum()
            val = objective(w)
            if (best_val is None) or (val < best_val):
                best_val = val
                best_w = w.copy()

    uni = np.ones(m) / m
    best_w = (1.0 - shrink) * best_w + shrink * uni
    best_w = np.clip(best_w, 0.0, 1.0)
    best_w = best_w / best_w.sum()
    return best_w


M = len(family_names)
weights = np.zeros((4, M), dtype=float)

# Optimize prob_12h for ranking.
def cidx_objective(w):
    p = np.zeros(len(train), dtype=float)
    for i in range(M):
        p += w[i] * stack_oof[i, :, 0]
    return -harrell_c_index(TARGET_TIME, TARGET_EVENT, p)

weights[0] = optimize_simplex(cidx_objective, M, shrink=0.25)

# Optimize 24/48/72 for Brier score.
for h_idx, horizon in enumerate(HORIZONS[1:], start=1):
    valid = horizon_valid_mask(TARGET_TIME, TARGET_EVENT, horizon)
    y = horizon_target(TARGET_TIME, TARGET_EVENT, horizon)[valid].astype(float)

    def brier_objective(w, idx=h_idx, valid_mask=valid, y_true=y):
        p = np.zeros(valid_mask.sum(), dtype=float)
        for i in range(M):
            p += w[i] * stack_oof[i, valid_mask, idx]
        return float(np.mean((np.clip(p, 1e-6, 1 - 1e-6) - y_true) ** 2) + 5e-4 * np.sum((w - 1.0 / M) ** 2))

    weights[h_idx] = optimize_simplex(brier_objective, M, shrink=0.25)

raw_oof = np.zeros((len(train), 4), dtype=float)
raw_test = np.zeros((len(test), 4), dtype=float)
for h_idx in range(4):
    for m_idx in range(M):
        raw_oof[:, h_idx] += weights[h_idx, m_idx] * stack_oof[m_idx, :, h_idx]
        raw_test[:, h_idx] += weights[h_idx, m_idx] * stack_test[m_idx, :, h_idx]

raw_oof = enforce_monotonic(raw_oof)
raw_test = enforce_monotonic(raw_test)

final_oof = raw_oof.copy()
final_test = raw_test.copy()

# Final calibration for the Brier horizons only.
for h_idx, horizon in enumerate(HORIZONS[1:], start=1):
    valid = horizon_valid_mask(TARGET_TIME, TARGET_EVENT, horizon)
    y = horizon_target(TARGET_TIME, TARGET_EVENT, horizon)[valid]
    final_oof[:, h_idx], final_test[:, h_idx] = beta_calibrate_probs(
        final_oof[:, h_idx], final_test[:, h_idx], valid, y
    )

final_oof = enforce_monotonic(final_oof)
final_test = enforce_monotonic(final_test)

print("\nBlend weights:")
for h_idx, horizon in enumerate(HORIZONS):
    print(f"\nH={horizon}h")
    for name, w in sorted(zip(family_names, weights[h_idx]), key=lambda x: x[1], reverse=True):
        print(f"  {name:>12s} : {w:.4f}")

print("\nOOF scores:")
print(f"Raw blend       : {hybrid_score(TARGET_TIME, TARGET_EVENT, raw_oof):.6f}")
print(f"Calibrated blend: {hybrid_score(TARGET_TIME, TARGET_EVENT, final_oof):.6f}")

LB_SCORE_MAP = {
    "samplecsv_sub1.csv": 0.96617,
    "samplecsv_sub2.csv": 0.96540,
    "samplecsv_sub3.csv": 0.96961,
    "samplecsv_sub4.csv": 0.97252,
    "samplecsv_sub5.csv": 0.97167,
    "samplecsv_sub6.csv": 0.97204,
    "samplecsv_sub7.csv": 0.97252,
    "samplecsv_sub8.csv": 0.97058,
    "samplecsv_sub9.csv": 0.95804,
    "samplecsv_sub10.csv": 0.97118,
    "samplecsv_sub11.csv": 0.96933,
    "samplecsv_sub12.csv": 0.96539,
    "samplecsv_sub13.csv": 0.96400,
    "samplecsv_sub14.csv": 0.95893,
    "samplecsv_sub15.csv": 0.96617,
    "samplecsv_sub16.csv": 0.96539,
    "samplecsv_sub17.csv": 0.97252,
    "samplecsv_sub18.csv": 0.96002,
    "samplecsv_sub19.csv": 0.95628,
    "samplecsv_sub20.csv": 0.96216,
    "samplecsv_sub21.csv": 0.95357,
    "samplecsv_sub22.csv": 0.97252,
    "samplecsv_sub23.csv": 0.97191,
    "samplecsv_sub24.csv": 0.95820,
    "samplecsv_sub25.csv": 0.95868,
}


def try_parse_score_from_name(name):
    m = re.search(r'_(0\.\d+)\.csv$', name)
    if m:
        try:
            return float(m.group(1))
        except Exception:
            return None
    return None



def load_external_submission_blend(sample_submission_df):
    root = Path("/kaggle/input")
    if not root.exists():
        return None

    candidates = []
    for p in root.rglob("*.csv"):
        name = p.name
        if name == "sample_submission.csv":
            continue
        score = LB_SCORE_MAP.get(name, try_parse_score_from_name(name))
        if score is None:
            continue
        try:
            df = pd.read_csv(p)
        except Exception:
            continue
        if not {"event_id", "prob_12h", "prob_24h", "prob_48h", "prob_72h"}.issubset(df.columns):
            continue
        if len(df) != len(sample_submission_df):
            continue

        df = sample_submission_df[["event_id"]].merge(
            df[["event_id"] + PROB_COLS],
            on="event_id",
            how="left",
        )
        if df[PROB_COLS].isna().any().any():
            continue

        preds = df[PROB_COLS].to_numpy(dtype=float)
        preds = enforce_monotonic(preds)
        candidates.append((name, score, preds))

    if len(candidates) < 2:
        return None

    candidates = sorted(candidates, key=lambda x: x[1], reverse=True)

    kept = []
    kept_flat = []
    for name, score, preds in candidates:
        flat = preds.ravel()
        if len(kept_flat) == 0:
            kept.append((name, score, preds))
            kept_flat.append(flat)
        else:
            corrs = []
            for prev in kept_flat:
                c = np.corrcoef(flat, prev)[0, 1]
                if np.isnan(c):
                    c = 1.0
                corrs.append(c)
            if max(corrs) < 0.9997:
                kept.append((name, score, preds))
                kept_flat.append(flat)
        if len(kept) >= 10:
            break

    scores = np.array([x[1] for x in kept], dtype=float)
    weights = np.exp((scores - scores.max()) / 0.0015)
    weights = weights / weights.sum()

    blend = np.zeros_like(kept[0][2], dtype=float)
    for w, (_, _, preds) in zip(weights, kept):
        blend += w * preds

    print("\nLoaded external submission blend:")
    for (name, score, _), w in zip(kept, weights):
        print(f"  {name:>18s} | score={score:.5f} | w={w:.4f}")

    return enforce_monotonic(blend)


if USE_EXTERNAL_SUBMISSION_BLEND:
    ext_blend = load_external_submission_blend(sample_sub)
    if ext_blend is not None:
        alpha = EXTERNAL_BLEND_MAX_WEIGHT
        final_test = enforce_monotonic((1.0 - alpha) * final_test + alpha * ext_blend)
        print(f"\nApplied external blend with alpha={alpha:.3f}")

submission = sample_sub[["event_id"]].copy()
submission[PROB_COLS] = enforce_monotonic(final_test)

assert submission["event_id"].tolist() == sample_sub["event_id"].tolist()
assert submission["event_id"].is_unique
assert np.isfinite(submission[PROB_COLS].to_numpy()).all()
assert ((submission[PROB_COLS] >= 0.0) & (submission[PROB_COLS] <= 1.0)).to_numpy().all()
assert (submission["prob_12h"] <= submission["prob_24h"]).all()
assert (submission["prob_24h"] <= submission["prob_48h"]).all()
assert (submission["prob_48h"] <= submission["prob_72h"]).all()

submission.to_csv("/kaggle/working/submission.csv", index=False)
submission.to_csv("submission.csv", index=False)

with open("/kaggle/working/model_summary.json", "w") as f:
    json.dump(
        {
            "family_names": family_names,
            "weights": {f"prob_{H}h": weights[i].tolist() for i, H in enumerate(HORIZONS)},
            "raw_oof_hybrid": hybrid_score(TARGET_TIME, TARGET_EVENT, raw_oof),
            "calibrated_oof_hybrid": hybrid_score(TARGET_TIME, TARGET_EVENT, final_oof),
        },
        f,
        indent=2,
    )

print("\nSaved: /kaggle/working/submission.csv")
print(submission.head(10).to_string(index=False))

gc.collect()


Training: direct_cb
Training: direct_xgb
Training: direct_lr
Training: direct_hgb
Training: hazard_cb
Training: hazard_lr
Training: urgency_cb
Training: urgency_hgb

OOF family scores:
   direct_cb : 0.967925
  direct_xgb : 0.966738
   direct_lr : 0.968423
  direct_hgb : 0.970108
   hazard_cb : 0.966531
   hazard_lr : 0.968691
  urgency_cb : 0.970686
 urgency_hgb : 0.972412

Blend weights:

H=12h
   urgency_hgb : 0.7811
     hazard_lr : 0.0314
     direct_lr : 0.0313
    urgency_cb : 0.0313
     direct_cb : 0.0313
    direct_xgb : 0.0313
    direct_hgb : 0.0313
     hazard_cb : 0.0313

H=24h
    direct_hgb : 0.4455
   urgency_hgb : 0.3670
     direct_cb : 0.0313
     hazard_cb : 0.0313
     hazard_lr : 0.0313
    direct_xgb : 0.0312
     direct_lr : 0.0312
    urgency_cb : 0.0312

H=48h
   urgency_hgb : 0.5443
    direct_hgb : 0.2639
    urgency_cb : 0.0355
     direct_cb : 0.0312
    direct_xgb : 0.0312
     direct_lr : 0.0312
     hazard_cb : 0.0312
     hazard_lr : 0.0312

H=72h
   

201